In [15]:
import requests
import pandas as pd

# A session makes sure our python script remembers things we did previously, like getting a cookie
# Without a session, we wouldn't retain the cookie we got from the front page and then we'd turn up to the API empty-handed
session = requests.Session()
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",  # this is a standard web browser fingerprint
    "X-Requested-With": "XMLHttpRequest",  # makes it look like it's the website's JavaScript itself fetching the data to populate the page
}
# We don't need anything from the front page but we want to look like a normal human who first went to the
# front page, got a cookie, then our browser showed the cookie to the API and called it
print("Step 1: Establishing Session & Pulling Payload...")
session.get("https://understat.com/league/EPL", headers=headers)
# This is getting the actual data from the hidden football data API
response = session.get("https://understat.com/getLeagueData/EPL/2025", headers=headers)

if response.status_code == 200:  # if we get in successfully:
    raw_data = response.json()

    # This extractor will still work even if understats adds a new layer to their JSON data,
    # because it searches recursively until it finds what we want
    def find_teams(data_node):
        teams = []
        #
        if isinstance(data_node, dict):  # if data_node is a dictionary:
            if (
                "title" in data_node and "history" in data_node
            ):  # if data_node has keys "title" and "history":
                teams.append(data_node)
            else:
                for (
                    value
                ) in (
                    data_node.values()
                ):  # for every single value inside this dictionary (whether it's another dict, a list, a string, or a number):
                    teams.extend(
                        find_teams(value)
                    )  # recursively calls find_teams until it finds the data we want, then puts everything in there into the array we return
        elif isinstance(data_node, list):  # if data_node is a list:
            for item in data_node:  # for each item in this list:
                teams.extend(
                    find_teams(item)
                )  # recursively call find_teams until it finds the data we want, then puts everything in there into the array we return
        return teams

    print("Step 2: Executing Recursive Search to bypass the wrappers...")
    extracted_teams = find_teams(raw_data)

    league_data = []
    for team in extracted_teams:
        team_name = team["title"]

        # Find home and away match data for each team
        home_matches = [match for match in team["history"] if match["h_a"] == "h"]
        away_matches = [match for match in team["history"] if match["h_a"] == "a"]

        # Find home/away xG/xGA for each team
        # Note that Understat sometimes stores xG/xGA as strings, so we convert to float to be safe
        home_xg = sum([float(match["xG"]) for match in home_matches])
        home_xga = sum([float(match["xGA"]) for match in home_matches])

        away_xg = sum([float(match["xG"]) for match in away_matches])
        away_xga = sum([float(match["xGA"]) for match in away_matches])

        league_data.append(
            {
                "Team": team_name,
                "Home_Matches": len(home_matches),
                "Away_Matches": len(away_matches),
                "Total_Matches": len(home_matches) + len(away_matches),
                "Home_xG": round(home_xg, 2),
                "Home_xGA": round(home_xga, 2),
                "Away_xG": round(away_xg, 2),
                "Away_xGA": round(away_xga, 2),
            }
        )

    df = pd.DataFrame(league_data)

    print("\nV2.0 Data Extraction Complete. Displaying Venue Splits:")
    display(df.head())

else:
    print("API Error.")

Step 1: Establishing Session & Pulling Payload...
Step 2: Executing Recursive Search to bypass the wrappers...

V2.0 Data Extraction Complete. Displaying Venue Splits:


,Team,Home_Matches,Away_Matches,Total_Matches,Home_xG,Home_xGA,Away_xG,Away_xGA
0,Aston Villa,14,13,27,19.42,16.41,17.21,23.52
1,Everton,14,13,27,19.35,20.84,14.79,21.14
2,Bournemouth,13,14,27,20.40,9.53,25.13,30.82
3,Sunderland,14,13,27,16.40,20.50,11.76,23.53
4,Crystal Palace,14,13,27,27.16,21.27,19.18,18.84


In [ ]:
# 1. Calculate Per-Game Averages
df["Home_xG_per_game"] = df["Home_xG"] / df["Home_Matches"]
df["Home_xGA_per_game"] = df["Home_xGA"] / df["Home_Matches"]

df["Away_xG_per_game"] = df["Away_xG"] / df["Away_Matches"]
df["Away_xGA_per_game"] = df["Away_xGA"] / df["Away_Matches"]

# 2. Calculate ALL 4 League Averages
# (Even though the pairs equal the exact same number, it makes the math clearer)
league_avg_home_xg = df["Home_xG_per_game"].mean()  # What avg home team scores
league_avg_away_xga = df["Away_xGA_per_game"].mean()  # What avg away team concedes

league_avg_away_xg = df["Away_xG_per_game"].mean()  # What avg away team scores
league_avg_home_xga = df["Home_xGA_per_game"].mean()  # What avg home team concedes

# 3. Calculate the 4 Core Strengths using your logic
# Home Attack: Team's Home xG relative to what the average Away Defense concedes
# this would be the same as doing team's Home xG relative to how many goals the average team scores at home,
# since home xG = away xGA over a full season. Same for all the below ratios.
df["Home_Attack"] = df["Home_xG_per_game"] / league_avg_away_xga

# Away Attack: Team's Away xG relative to what the average Home Defense concedes
df["Away_Attack"] = df["Away_xG_per_game"] / league_avg_home_xga

# Home Defense: Team's Home xGA relative to what the average Away Attack scores
df["Home_Defense"] = df["Home_xGA_per_game"] / league_avg_away_xg

# Away Defense: Team's Away xGA relative to what the average Home Attack scores
df["Away_Defense"] = df["Away_xGA_per_game"] / league_avg_home_xg

print("\nV2.0 Model Parameters Calculated (Using Explicit Logic):")
display(
    df[["Team", "Home_Attack", "Home_Defense", "Away_Attack", "Away_Defense"]].head(10)
)


V2.0 Model Parameters Calculated (Using Explicit Logic):


,Team,Home_Attack,Home_Defense,Away_Attack,Away_Defense
0,Aston Villa,0.831370,0.870083,0.983965,1.084085
1,Everton,0.828373,1.104969,0.845604,0.974386
2,Bournemouth,0.940502,0.544164,1.334156,1.319089
3,Sunderland,0.702084,1.086941,0.672366,1.084546
4,Crystal Palace,1.162719,1.127768,1.096598,0.868374
5,Chelsea,1.284728,1.144735,1.532263,0.852242
6,West Ham,0.835223,1.233811,0.873047,1.266149
7,Tottenham,0.746606,1.154809,0.830167,0.885428
8,Arsenal,1.247549,0.517898,1.463731,0.500129
9,Newcastle United,1.378482,1.008959,0.765560,0.764832


In [12]:
def calculate_match_lambdas(home_team, away_team, df, league_avg_xg):

    # IMPORTANT: These home_team and away_team variables DO NOT take into account home advantage. They are not xG
    # for that team at home, they are just naming conventions for the two teams so we can distinguish them

    # we will address this later to account for home advantage

    # 1. Extract the specific stats for the Home Team
    home_stats = df[df["Team"] == home_team].iloc[0]
    home_attack = home_stats["Attack_Strength"]
    home_defense = home_stats["Defense_Strength"]

    # 2. Extract the specific stats for the Away Team
    away_stats = df[df["Team"] == away_team].iloc[0]
    away_attack = away_stats["Attack_Strength"]
    away_defense = away_stats["Defense_Strength"]

    # 3. Calculate Lambda (Expected Goals for this specific match)
    # Home Expected Goals = Home Attack * Away Defense * League Average
    home_lambda = home_attack * away_defense * league_avg_xg

    # Away Expected Goals = Away Attack * Home Defense * League Average
    away_lambda = away_attack * home_defense * league_avg_xg

    return round(home_lambda, 3), round(away_lambda, 3)


# --- Test it out ---
# Make sure 'league_avg_xg_per_game' is defined from your previous cell
home_team_test = "Burnley"  # Replace with a valid name from your 'Team' column
away_team_test = "Liverpool"

home_xg, away_xg = calculate_match_lambdas(
    home_team_test, away_team_test, df, league_avg_xg_per_game
)

print(
    f"Projected Matchup: {home_team_test} ({home_xg} xG) vs {away_team_test} ({away_xg} xG)"
)

Projected Matchup: Burnley (0.821 xG) vs Liverpool (2.678 xG)


In [13]:
import numpy as np
from scipy.stats import poisson
import pandas as pd


def calculate_match_probabilities(home_lambda, away_lambda, max_goals=5):
    # 1. Create an array of possible goals (0 through 5)
    goals_range = np.arange(0, max_goals + 1)  # creates an array from 0 to max_goals

    # 2. Calculate the independent Poisson probabilities for each team
    # poisson.pmf (Probability Mass Function) takes the goal range and the lambda
    home_probs = poisson.pmf(
        goals_range, home_lambda
    )  # creates an array of probs for each number in goals_range
    away_probs = poisson.pmf(goals_range, away_lambda)

    # 3. Build the Bivariate Matrix
    # np.outer multiplies every item in home_probs by every item in away_probs
    prob_matrix = np.outer(
        home_probs, away_probs
    )  # creates a 6x6 matrix for every combination of goals and says their probabilities

    # 4. Sum the matrix sections to get Match Odds
    home_win_prob = np.sum(
        np.tril(prob_matrix, k=-1)
    )  # tril means triangle-lower, setting k=-1 means we only keep everything below the main diagonal,
    # so where Home Goals > Away Goals

    # np.diag gets the main diagonal (Home Goals == Away Goals)
    draw_prob = np.sum(np.diag(prob_matrix))

    # np.triu(matrix, 1) gets everything above the diagonal (Home Goals < Away Goals)
    away_win_prob = np.sum(np.triu(prob_matrix, 1))

    # Let's wrap the matrix in a Pandas DataFrame so it looks nice if you print it
    matrix_df = pd.DataFrame(
        prob_matrix,
        columns=[f"Away_{i}" for i in range(max_goals + 1)],
        index=[f"Home_{i}" for i in range(max_goals + 1)],
    )

    return {
        "Home_Win_Prob": home_win_prob,
        "Draw_Prob": draw_prob,
        "Away_Win_Prob": away_win_prob,
        "Matrix": matrix_df,
    }


# --- Test the Engine ---
# We'll pass in the home_xg and away_xg you calculated in the previous step
match_results = calculate_match_probabilities(home_xg, away_xg, max_goals=5)

print("--- Model Predictions ---")
print(f"Home Win:  {match_results['Home_Win_Prob'] * 100:.2f}%")
print(f"Draw:      {match_results['Draw_Prob'] * 100:.2f}%")
print(f"Away Win:  {match_results['Away_Win_Prob'] * 100:.2f}%\n")

print("--- Exact Scoreline Probability Matrix ---")
# Multiply by 100 and round to 2 decimals for readability
display(match_results["Matrix"].map(lambda x: f"{x*100:.2f}%"))

--- Model Predictions ---
Home Win:  8.97%
Draw:      14.35%
Away Win:  71.16%

--- Exact Scoreline Probability Matrix ---


,Away_0,Away_1,Away_2,Away_3,Away_4,Away_5
Home_0,3.02%,8.09%,10.84%,9.68%,6.48%,3.47%
Home_1,2.48%,6.65%,8.90%,7.94%,5.32%,2.85%
Home_2,1.02%,2.73%,3.65%,3.26%,2.18%,1.17%
Home_3,0.28%,0.75%,1.00%,0.89%,0.60%,0.32%
Home_4,0.06%,0.15%,0.21%,0.18%,0.12%,0.07%
Home_5,0.01%,0.03%,0.03%,0.03%,0.02%,0.01%


In [14]:
import pandas as pd


def find_value_bets(match_results, bookmaker_odds):
    bets_analysis = []

    # Map the bookmaker keys to your model's probability keys
    outcomes = {"Home": "Home_Win_Prob", "Draw": "Draw_Prob", "Away": "Away_Win_Prob"}

    for outcome, prob_key in outcomes.items():
        # Get your model's true probability
        model_prob = match_results[prob_key]

        # Convert to implied decimal odds (handle division by zero just in case)
        model_odds = 1 / model_prob if model_prob > 0 else float("inf")

        # Get the bookmaker's odds
        bookie_odds = bookmaker_odds.get(outcome, 0)

        # Calculate Expected Value (EV)
        # EV = (True Probability * Bookmaker Decimal Odds) - 1
        ev = (model_prob * bookie_odds) - 1

        bets_analysis.append(
            {
                "Market": outcome,
                "Model_Prob": f"{model_prob * 100:.2f}%",
                "Model_Odds": round(model_odds, 2),
                "Bookie_Odds": bookie_odds,
                "Edge (EV)": f"{ev * 100:.2f}%",
                "Bet_Signal": "🔥 VALUE (BET)" if ev > 0 else "PASS",
            }
        )

    return pd.DataFrame(bets_analysis)


# --- Test the Arbitrage Scanner ---
# Let's pretend Pinnacle has these odds for the match:
# Home: 2.10, Draw: 3.50, Away: 3.20
pinnacle_odds = {"Home": 2.10, "Draw": 3.50, "Away": 3.20}

# Pass in the 'match_results' dictionary we generated in Step 3
value_dashboard = find_value_bets(match_results, pinnacle_odds)

print("--- Market Analysis Dashboard ---")
display(value_dashboard)

--- Market Analysis Dashboard ---


,Market,Model_Prob,Model_Odds,Bookie_Odds,Edge (EV),Bet_Signal
0,Home,8.97%,11.15,2.1,-81.16%,PASS
1,Draw,14.35%,6.97,3.5,-49.78%,PASS
2,Away,71.16%,1.41,3.2,127.72%,🔥 VALUE (BET)
